# GxP-LLM fine-tuning

Attach versioned `gxp-source` and `gxp-data`, enable Internet and GPU, then save the completed notebook version. Its `/kaggle/working/gxp_train` directory becomes the input model artifact for quantization.

In [ ]:
%pip install -q trl peft bitsandbytes accelerate datasets wandb sentence-transformers rouge-score nltk litellm
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print(torch.cuda.get_device_name(0))

In [ ]:
import json, os, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import wandb

SOURCE_DIR, DATA_DIR = Path('/kaggle/input/gxp-source'), Path('/kaggle/input/gxp-data')
assert (SOURCE_DIR / 'eval' / 'run.py').exists() and (DATA_DIR / 'train.jsonl').exists(), 'Attach gxp-source and gxp-data.'
sys.path.insert(0, str(SOURCE_DIR))
OUTPUT_DIR = Path('/kaggle/working/gxp_train'); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = 'unsloth/Qwen3.5-4B'
JUDGE_MODEL = 'gemini/gemini-2.5-flash'

def load_secret(name):
    value = UserSecretsClient().get_secret(name)
    if not value: raise RuntimeError(f'Add {name} as a Kaggle Secret.')
    os.environ[name] = value

load_secret('WANDB_API_KEY')
load_secret({'gemini/': 'GEMINI_API_KEY', 'groq/': 'GROQ_API_KEY', 'nvidia_nim/': 'NVIDIA_API_KEY'}[next(k for k in ('gemini/', 'groq/', 'nvidia_nim/') if JUDGE_MODEL.startswith(k))])
config = {'stage':'finetuned','model':MODEL_ID,'method':'QLoRA 4-bit NF4','rank':16,'alpha':16,'max_seq_length':2048,'batch_size':2,'grad_accum':4,'learning_rate':2e-4,'max_steps':300,'judge_model':JUDGE_MODEL}
run = wandb.init(project='gxp-llm', name=f'finetune-{MODEL_ID.rsplit("/", 1)[-1]}-r16', group=MODEL_ID.rsplit('/', 1)[-1], job_type='finetune', tags=['finetuned','qlora'], config=config)
(OUTPUT_DIR / 'experiment_config.json').write_text(json.dumps(config, indent=2))

In [ ]:
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer

def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines()]
def format_chat(ex):
    return {'text': '<|im_start|>system\n' + ex['messages'][0]['content'] + '<|im_end|>\n<|im_start|>user\n' + ex['messages'][1]['content'] + '<|im_end|>\n<|im_start|>assistant\n' + ex['messages'][2]['content'] + '<|im_end|>'}

train_ds = Dataset.from_list([format_chat(x) for x in load_jsonl(DATA_DIR / 'train.jsonl')])
eval_ds = Dataset.from_list([format_chat(x) for x in load_jsonl(DATA_DIR / 'eval.jsonl')])
model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_ID, max_seq_length=2048, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
FastLanguageModel.for_training(model)

trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=eval_ds, args=SFTConfig(output_dir=str(OUTPUT_DIR / 'checkpoints'), per_device_train_batch_size=2, gradient_accumulation_steps=4, max_steps=300, learning_rate=2e-4, logging_steps=10, eval_strategy='steps', eval_steps=50, save_strategy='steps', save_steps=50, save_total_limit=3, report_to='wandb', fp16=True, bf16=False, load_best_model_at_end=True, metric_for_best_model='eval_loss'))
trainer.train()

In [ ]:
# Publish portable outputs and run the same post-fine-tune evaluation.
model.save_pretrained_merged(str(OUTPUT_DIR / 'merged_16bit'), tokenizer, save_method='merged_16bit')
model.save_pretrained(str(OUTPUT_DIR / 'lora_adapter')); tokenizer.save_pretrained(str(OUTPUT_DIR / 'lora_adapter'))
from eval.run import run_full_eval
results = run_full_eval(model_path=str(OUTPUT_DIR / 'merged_16bit'), data_dir=str(DATA_DIR), judge_model=JUDGE_MODEL, output_dir=str(OUTPUT_DIR / 'eval_finetuned'), load_in_4bit=True)
artifact = wandb.Artifact(f'model-finetuned-{MODEL_ID.rsplit("/", 1)[-1]}', type='model', metadata=config); artifact.add_dir(str(OUTPUT_DIR)); wandb.log_artifact(artifact)
wandb.finish(); print(f'Published Kaggle output: {OUTPUT_DIR}')